# 02b — 지역 추가 (신규 7곳)

`02_regions`의 함수만 가져와 **신규 지역 전용**으로 만든 노트북. 02의 홍천·순천 셀(`max_km=999`)은 여기 없다.

| 역할 | 지역 (Drive 폴더) |
|---|---|
| 학습 (확정 8) | inje, hongcheon, gangneung, pyeongchang, bonghwa, geumsan, geochang, suncheon |
| 시험 | daejeon (도시 파편화) · wando (상록활엽 41% — 적용 범위 밖 확인) |

**실행 순서** — "모두 실행" 금지. 셀 하나씩 실행한다.

| 셀 | 내용 |
|---|---|
| `[1]` `[2]` | 셋업 · 함수 (세션 끊기면 항상 먼저) |
| `[3]` | `screen()` 검산 — 기존 3곳 ✅ 완료 |
| `[4]` | `screen()` 신규 7곳 ✅ 완료 → 학습 6곳 확정 |
| `[5]` | 완도 상록활엽 재확인 ✅ 완료 (41.4%) |
| `[6]` | 상록활엽 재계산 — 전 지역 (코드 60 포함) |
| `[7]` | export (진행 중인 작업은 자동으로 건너뜀) |
| `[8]` | export 상태 |
| `[9]` → `[10]` → `[11]` → `[12]` | 파일 이동 → 라벨 생성 → 픽셀 수 → 정합 검증 |
| `[13]` | 지역 기록표 |

**셀 번호 규칙** — 셀을 추가하면 끝에 다음 번호로 붙이거나, 중간에 넣을 땐 전체 번호를 다시 매긴다. `4-1` 같은 번호는 쓰지 않는다.

In [15]:
# ============================================================
# [1] 셋업 (02 [1]과 동일)
# ============================================================
!pip install -q earthengine-api geopandas pyogrio rasterio

import ee, numpy as np, pandas as pd, geopandas as gpd
import rasterio, pyogrio, os, glob, shutil
from rasterio.features import rasterize
from shapely.geometry import box
from google.colab import drive

PROJECT_ID = 'driven-era-467023-m3'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
D       = '/content/drive/MyDrive/forest/data'
SRC_DIR = '/content/drive/MyDrive/forest_s2'      # GEE export 도착 폴더
MAX_KM  = 35

BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
CLASS_MAP = {'0':0, '1':1, '2':2, '3':3, '4':0}   # 죽림은 배경 흡수

NEW = ['inje', 'pyeongchang', 'gangneung', 'bonghwa', 'geumsan', 'geochang', 'wando']
OLD = ['daejeon', 'hongcheon', 'suncheon']

def cloud_mask(img):
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def composite_aoi(aoi, start, end, months, cloud_pct=50):
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi).filterDate(start, end).filter(mf)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
            .map(cloud_mask).median().clip(aoi))

print('셋업 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
셋업 완료


In [16]:
# ============================================================
# [2] 함수
#   load_shp      : 폴더의 shp 1세트 읽기 (2세트 이상이면 중단)
#   aoi_bounds    : export_region 과 똑같은 방식으로 AOI(경위도 사각형) 계산
#   screen        : export 전 사전 선별 — AOI 안의 2024+ 라벨만 센다
#   export_region : 02 [2]와 동일 + 영상 장수를 기록용으로 반환
#   build_labels  : 02 [10]과 동일
#   check_align   : 02 [13]과 동일
# ============================================================
_SHP = {}
def load_shp(name):
    if name in _SHP: return _SHP[name]
    shps = glob.glob(f'{D}/imsang/2025_{name}/*.shp')
    assert len(shps) == 1, f'{name}: shp {len(shps)}개 — 폴더에 1세트만 있어야 함'
    gdf = gpd.read_file(shps[0], engine='pyogrio')
    for col in ['FRTP_CD', '갱신년도', 'KOFTR_GROU']:
        assert col in gdf.columns, f'{name}: {col} 열 없음 → 열 이름 확인 필요: {list(gdf.columns)}'
    _SHP[name] = gdf
    return gdf


def aoi_bounds(gdf, max_km=MAX_KM):
    b = gdf.to_crs(4326).total_bounds
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    dx = min((b[2]-b[0])/2, max_km/2/88)      # 경도 1도 ≈ 88km
    dy = min((b[3]-b[1])/2, max_km/2/111)     # 위도 1도 ≈ 111km
    return cx-dx, cy-dy, cx+dx, cy+dy


def screen(name, max_km=MAX_KM):
    g = load_shp(name)
    x0, y0, x1, y1 = aoi_bounds(g, max_km)
    clip_box = gpd.GeoSeries([box(x0, y0, x1, y1)], crs=4326).to_crs(g.crs).iloc[0]
    g = g[g.intersects(clip_box)]
    g = gpd.clip(g, clip_box)

    cd = g['FRTP_CD'].astype(str)
    yr = pd.to_numeric(g['갱신년도'], errors='coerce')
    f  = g[cd.isin(['1','2','3']) & (yr >= 2024)]
    a  = f.geometry.area
    fc = f['FRTP_CD'].astype(str)
    px = (a / 100).groupby(fc).sum()                      # 10m 픽셀 수
    sp = pd.to_numeric(f['KOFTR_GROU'], errors='coerce')
    con = fc == '1'
    larch = a[con & (sp == 13)].sum() / a[con].sum() * 100 if con.any() else 0
    ev    = a[sp.between(60, 68)].sum() / a.sum() * 100 if len(f) else 0   # 면적 기준

    aoi = ee.Geometry.Rectangle([x0, y0, x1, y1])
    elev = ee.Image('USGS/SRTMGL1_003').reduceRegion(
        ee.Reducer.mean(), aoi, scale=90, maxPixels=1e9).get('elevation').getInfo()

    tot = int(px.sum())
    return dict(name=name, lat=round((y0+y1)/2, 2), lon=round((x0+x1)/2, 2),
                aoi_km=f'{(x1-x0)*88:.0f}x{(y1-y0)*111:.0f}',
                conif=int(px.get('1', 0)), broad=int(px.get('2', 0)), mixed=int(px.get('3', 0)),
                total=tot,
                conif_pct=round(px.get('1', 0)/max(tot,1)*100, 1),
                larch_pct=round(larch, 1), evergreen_pct=round(ev, 2),
                elev_mean=round(elev, 0) if elev else None)


def export_region(name, max_km=MAX_KM, start='2024-01-01', end='2025-12-31'):
    gdf = load_shp(name)
    x0, y0, x1, y1 = aoi_bounds(gdf, max_km)
    aoi = ee.Geometry.Rectangle([x0, y0, x1, y1])
    print(f'{name} AOI : {(x1-x0)*88:.0f} x {(y1-y0)*111:.0f} km')

    summer  = composite_aoi(aoi, start, end, [6,7,8,9])
    leafoff = composite_aoi(aoi, start, end, [11,12,1,2])

    n_img = {}
    for nm, mo in [('summer',[6,7,8,9]), ('leafoff',[11,12,1,2])]:
        mf = ee.Filter.Or([ee.Filter.calendarRange(m,m,'month') for m in mo])
        n_img[nm] = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                     .filterBounds(aoi).filterDate(start,end).filter(mf)
                     .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',50)).size().getInfo())
    print(f'  영상 장수  여름 {n_img["summer"]} / 낙엽기 {n_img["leafoff"]}')

    stack = (summer.select(BANDS).rename([f's_{b}' for b in BANDS])
             .addBands(leafoff.select(BANDS).rename([f'w_{b}' for b in BANDS])))
    stack = stack.multiply(10000).toInt16()             # int16 — 읽을 때 /10000

    task = ee.batch.Export.image.toDrive(
        image=stack, description=f'{name}_s2',
        folder='forest_s2', fileNamePrefix=f'{name}_s2_20band',
        region=aoi, scale=10, crs=str(gdf.crs), maxPixels=1e10)
    task.start()
    print('  export 제출')
    return task, n_img


def build_labels(name):
    gdf = load_shp(name)
    s2 = f'{D}/s2/{name}_s2_20band.tif'
    with rasterio.open(s2) as src:
        meta, transform = src.meta.copy(), src.transform
        shape, s2_crs   = (src.height, src.width), src.crs

    g = gdf.to_crs(s2_crs) if gdf.crs != s2_crs else gdf.copy()
    g['cls'] = g['FRTP_CD'].astype(str).map(CLASS_MAP).fillna(0).astype('uint8')
    g['yr']  = pd.to_numeric(g['갱신년도'],   errors='coerce').fillna(0).astype('uint16')
    g['sp']  = pd.to_numeric(g['KOFTR_GROU'], errors='coerce').fillna(0).astype('uint16')

    for col, fill, dt, suffix in [('cls', 255, 'uint8',  'label'),
                                  ('yr',    0, 'uint16', 'year'),
                                  ('sp',    0, 'uint16', 'species')]:
        arr = rasterize(((geom, v) for geom, v in zip(g.geometry, g[col])),
                        out_shape=shape, transform=transform, fill=fill, dtype=dt)
        m = meta.copy()
        m.update(count=1, dtype=dt, nodata=fill, compress='lzw')
        with rasterio.open(f'{D}/{name}_{suffix}.tif', 'w', **m) as dst:
            dst.write(arr, 1)
        if col == 'cls': lab = arr
    u, c = np.unique(lab, return_counts=True)
    print(f'  {name} 라벨 생성 완료  ' + '  '.join(f'{v}:{n:,}' for v, n in zip(u, c)))


def check_align(name, sz=1200):
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from rasterio.windows import Window
    with rasterio.open(f'{D}/{name}_label.tif') as s: label = s.read(1)
    v = np.argwhere(label != 255)
    cy, cx = v.mean(axis=0).astype(int)
    r0 = max(0, min(cy - sz//2, label.shape[0] - sz))
    c0 = max(0, min(cx - sz//2, label.shape[1] - sz))
    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as src:
        rgb = src.read([3, 2, 1], window=Window(c0, r0, sz, sz)).astype('float32')
        if src.dtypes[0] == 'int16': rgb /= 10000.0
    rgb = np.clip((rgb - 0.02) / 0.23, 0, 1).transpose(1, 2, 0)
    lab_m = np.ma.masked_where(label[r0:r0+sz, c0:c0+sz] == 255, label[r0:r0+sz, c0:c0+sz])
    cmap = ListedColormap(['#bbbbbb', '#1b7837', '#d95f02', '#e7c419'])
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(rgb); ax[0].set_title(f'{name} RGB')
    ax[1].imshow(rgb); ax[1].imshow(lab_m, cmap=cmap, vmin=0, vmax=3, alpha=0.45)
    ax[1].set_title('overlay  회색=배경 녹=침엽 주황=활엽 노랑=혼효')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

print('함수 준비 완료')

함수 준비 완료


In [ ]:
# ============================================================
# [3] screen() 검산 — 기존 3곳
#   기존 3곳은 행정구역 전체(max_km=999)로 export 했으므로 같은 조건으로 센다.
#   기대값 (02 [12], 래스터 기준 2024+ 픽셀):
#     daejeon 1,029,600 / hongcheon 4,737,009 / suncheon 1,867,787
#   폴리곤 면적 기준이라 약간 클 수 있다. 수 % 이내면 통과.
#   larch_pct 는 03 [10](8.6 / 33 / 0, 학습 샘플 기준)과 '비슷한 수준'이면 통과.
# ============================================================
EXPECT = {'daejeon': 1_029_600, 'hongcheon': 4_737_009, 'suncheon': 1_867_787}
rows = []
for n in OLD:
    r = screen(n, max_km=999)
    r['expect'] = EXPECT[n]
    r['diff_pct'] = round((r['total'] / EXPECT[n] - 1) * 100, 1)
    rows.append(r)
chk = pd.DataFrame(rows)
print(chk[['name','total','expect','diff_pct','larch_pct','conif_pct','lat']].to_string(index=False))

     name   total  expect  diff_pct  larch_pct  conif_pct   lat
  daejeon 1029501 1029600      -0.0        8.6       39.8 36.34
hongcheon 4736780 4737009      -0.0       37.4       37.7 37.75
 suncheon 1867840 1867787       0.0        0.0       51.9 35.01


In [ ]:
# ============================================================
# [4] screen() 신규 7곳 → screen_result.csv
#   export 와 같은 35km 범위로 잘라서 센다.
#   ★ 이 출력을 Claude에게 붙여넣고 지역을 확정한 뒤 [5]로 넘어갈 것
#
#   선정 기준 (인수인계 19-6, 사전 확정)
#     1. total(2024+ 픽셀) ≥ 600,000
#     2. 낙엽송 비율이 위도와 같은 순서로 줄지 않을 것
#        — 북쪽 저낙엽송 또는 남쪽 고낙엽송 최소 1곳
#     3. 해안 지역 최소 1곳 (강릉)
# ============================================================
rows = []
for n in NEW:
    try:
        rows.append(screen(n)); print(f'{n} 완료')
    except Exception as e:
        print(f'{n} 실패: {e}')
res = pd.DataFrame(rows).sort_values('lat', ascending=False)
res['ok_600k'] = res['total'] >= 600_000
res.to_csv(f'{D}/screen_result.csv', index=False)
print()
print(res.to_string(index=False))

inje 완료
pyeongchang 완료
gangneung 완료
bonghwa 완료
geumsan 완료
geochang 완료
wando 완료

       name   lat    lon aoi_km   conif   broad  mixed   total  conif_pct  larch_pct  evergreen_pct  elev_mean  ok_600k
       inje 38.04 128.25  35x35 1685673 1561568 477794 3725035       45.3       19.0           0.00      622.0     True
  gangneung 37.71 128.83  35x35 1185042 1421360 317692 2924095       40.5       11.0           0.00      434.0     True
pyeongchang 37.54 128.50  35x35 1516749 1585289 331632 3433670       44.2       45.8           0.00      760.0     True
    bonghwa 36.92 128.91  35x35 2304358 1257247 264193 3825800       60.2       16.7           0.00      561.0     True
    geumsan 36.13 127.48  28x33  278007  439896 138604  856508       32.5       19.5           0.00      284.0     True
   geochang 35.71 127.89  35x35  766066  450554 306367 1522988       50.3       11.7           0.00      550.0     True
      wando 34.21 126.79  35x35  124024  179713   7893  311632       39.8       

In [ ]:
# ============================================================
# [5] 완도 상록활엽 재확인 — 코드 60(기타상록활엽) 포함
#   결과: 2024+ 산림의 41.4% 가 상록활엽 → 시험 지역으로 유지
# ============================================================
g = load_shp('wando')
x0, y0, x1, y1 = aoi_bounds(g)
cb = gpd.GeoSeries([box(x0, y0, x1, y1)], crs=4326).to_crs(g.crs).iloc[0]
g = gpd.clip(g[g.intersects(cb)], cb)

cd = g['FRTP_CD'].astype(str)
yr = pd.to_numeric(g['갱신년도'], errors='coerce')
sp = pd.to_numeric(g['KOFTR_GROU'], errors='coerce')
a  = g.geometry.area / 1e4                      # ha

for label, m in [('2024+ 산림', cd.isin(['1','2','3']) & (yr >= 2024)),
                 ('전체 연도 산림', cd.isin(['1','2','3']))]:
    tot = a[m].sum()
    ev  = a[m & sp.between(60, 68)].sum()
    print(f'{label:10s} 면적 {tot:8.0f} ha   상록활엽(60–68) {ev:7.0f} ha = {ev/tot*100:5.1f}%')

m = cd.isin(['2','3']) & (yr >= 2024)
print('\n2024+ 활엽·혼효림 수종 구성 (상위 10, ha)')
print((a[m].groupby(sp[m]).sum().sort_values(ascending=False).head(10)).round(0).to_string())

2024+ 산림   면적     3116 ha   상록활엽(60–68)    1289 ha =  41.4%
전체 연도 산림   면적    12368 ha   상록활엽(60–68)    2326 ha =  18.8%

2024+ 활엽·혼효림 수종 구성 (상위 10, ha)
KOFTR_GROU
60    1275.0
30     350.0
34     136.0
77      79.0
35      14.0
67       9.0
31       8.0
65       4.0
45       1.0
61       0.0


In [17]:
# ============================================================
# [6] 상록활엽 재계산 — 전 지역 (코드 60–68)
#   [4] 의 evergreen_pct 는 코드 60(기타상록활엽)을 빠뜨려 과소 추정됐다.
#   수종 코드는 라벨·실험에는 쓰이지 않으므로 결과에 영향 없음 — 기록표만 바로잡는다.
#   기존 3곳은 export 범위와 같게 행정구역 전체(999), 신규는 35km.
#   판단: 학습 지역에서 1% 미만이면 무시, 수 % 이상이면 기록표에 남기고 해석에 반영
# ============================================================
def evergreen_pct(name, max_km=MAX_KM):
    g = load_shp(name)
    x0, y0, x1, y1 = aoi_bounds(g, max_km)
    cb = gpd.GeoSeries([box(x0, y0, x1, y1)], crs=4326).to_crs(g.crs).iloc[0]
    g = gpd.clip(g[g.intersects(cb)], cb)
    cd = g['FRTP_CD'].astype(str)
    yr = pd.to_numeric(g['갱신년도'], errors='coerce')
    sp = pd.to_numeric(g['KOFTR_GROU'], errors='coerce')
    a  = g.geometry.area
    m  = cd.isin(['1','2','3']) & (yr >= 2024)
    return round(a[m & sp.between(60, 68)].sum() / a[m].sum() * 100, 2)

ev = {}
for n in OLD + NEW:
    ev[n] = evergreen_pct(n, max_km=999 if n in OLD else MAX_KM)
    print(f'{n:12s} 상록활엽 {ev[n]:6.2f}%')

res = pd.read_csv(f'{D}/screen_result.csv')
res['evergreen_pct'] = res['name'].map(ev)
res.to_csv(f'{D}/screen_result.csv', index=False)
pd.DataFrame({'name': OLD, 'evergreen_pct': [ev[n] for n in OLD]}).to_csv(
    f'{D}/evergreen_old_regions.csv', index=False)
print('\nscreen_result.csv 갱신 완료 (신규 7곳), 기존 3곳은 evergreen_old_regions.csv')

daejeon      상록활엽   0.00%
hongcheon    상록활엽   0.00%
suncheon     상록활엽   0.11%
inje         상록활엽   0.00%
pyeongchang  상록활엽   0.00%
gangneung    상록활엽   0.00%
bonghwa      상록활엽   0.00%
geumsan      상록활엽   0.00%
geochang     상록활엽   0.00%
wando        상록활엽  41.35%

screen_result.csv 갱신 완료 (신규 7곳), 기존 3곳은 evergreen_old_regions.csv


---
## 여기서부터 export 이후 단계 — 셀 하나씩 실행

`[7]`은 BATCH 만 바꿔서 실행한다. tif 가 있거나 GEE 에서 진행 중이면 자동으로 건너뛴다.

In [18]:
# ============================================================
# [7] export — 이번에 보낼 지역만 BATCH 에 넣는다
#   tif 가 이미 있거나, GEE 에서 진행 중(READY/RUNNING)이면 건너뛴다
# ============================================================
BATCH = ['wando']   # ← 여기만 바꿔서 실행
# 제출 이력: inje·pyeongchang (1차), gangneung·bonghwa·geumsan·geochang (2차) — wando 는 아직

LOG = f'{D}/export_log.csv'
active = {t['description'] for t in ee.data.getTaskList()
          if t.get('state') in ('READY', 'RUNNING')}
tasks = globals().get('tasks', {})
for n in BATCH:
    if os.path.exists(f'{D}/s2/{n}_s2_20band.tif') or glob.glob(f'{SRC_DIR}/{n}_s2_20band*.tif'):
        print(f'{n}: tif 이미 있음 → 건너뜀'); continue
    if f'{n}_s2' in active:
        print(f'{n}: GEE 에서 진행 중 → 건너뜀'); continue
    t, n_img = export_region(n)
    tasks[n] = t
    row = pd.DataFrame([dict(name=n, max_km=MAX_KM, summer=n_img['summer'],
                             leafoff=n_img['leafoff'], submitted=pd.Timestamp.now())])
    row.to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)

wando AOI : 35 x 35 km
  영상 장수  여름 142 / 낙엽기 120
  export 제출


In [37]:
# ============================================================
# [8] export 상태 (수시로 실행)
#   GEE 에 직접 조회하므로 Colab 세션이 끊겨도 동작한다 ([1] 만 실행돼 있으면 됨)
#   READY = 대기 중, RUNNING = 처리 중, COMPLETED = 완료 → [9] 진행 가능
# ============================================================
import logging
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)   # 무해한 경고 숨김

ALL = ['inje', 'pyeongchang', 'gangneung', 'bonghwa', 'geumsan', 'geochang', 'wando']
latest = {}
for t in ee.data.getTaskList():                      # 최신 작업이 먼저 나온다
    d = t.get('description', '')
    if d.endswith('_s2') and d[:-3] in ALL and d[:-3] not in latest:
        latest[d[:-3]] = t
for n in ALL:
    t = latest.get(n)
    if t is None:
        print(f'{n:12s} (작업 없음)'); continue
    st = t.get('state')
    mins = ''
    if t.get('start_timestamp_ms') and t.get('update_timestamp_ms'):
        mins = f"{(t['update_timestamp_ms'] - t['start_timestamp_ms']) / 60000:5.0f}분"
    err = f"  ← {t.get('error_message', '')}" if st == 'FAILED' else ''
    print(f'{n:12s} {st:10s} {mins}{err}')
done = [n for n in ALL if latest.get(n, {}).get('state') == 'COMPLETED']
print(f'\n완료 {len(done)}/7 : {done}')

inje         COMPLETED     18분
pyeongchang  COMPLETED     12분
gangneung    COMPLETED     21분
bonghwa      COMPLETED     14분
geumsan      COMPLETED     19분
geochang     COMPLETED      8분
wando        COMPLETED     19분

완료 7/7 : ['inje', 'pyeongchang', 'gangneung', 'bonghwa', 'geumsan', 'geochang', 'wando']


In [38]:
# ============================================================
# [9] forest_s2/ → data/s2/ 이동 + 지역별 상태
#   Drive 를 다시 연결해서 export 이후 새로 생긴 파일도 보이게 한다
#   GEE 가 큰 파일을 여러 조각으로 나누면 이름 뒤에 -0000000000-... 가 붙는다.
#   그 경우 이동하지 말고 Claude에게 알릴 것 (합치기 필요)
# ============================================================
drive.mount('/content/drive', force_remount=True)
os.makedirs(f'{D}/s2', exist_ok=True)

ALL = ['inje', 'pyeongchang', 'gangneung', 'bonghwa', 'geumsan', 'geochang', 'wando']
for n in ALL:
    dst  = f'{D}/s2/{n}_s2_20band.tif'
    src  = f'{SRC_DIR}/{n}_s2_20band.tif'
    part = glob.glob(f'{SRC_DIR}/{n}_s2_20band-*.tif')
    if os.path.exists(dst):
        print(f'{n:12s} 이동 완료(이전)   {os.path.getsize(dst)/1e6:5.0f} MB')
    elif part:
        print(f'{n:12s} 분할 파일 {len(part)}개 → 이동 보류, Claude에게 알릴 것')
    elif os.path.exists(src):
        shutil.move(src, dst)
        print(f'{n:12s} 지금 이동        {os.path.getsize(dst)/1e6:5.0f} MB')
    else:
        print(f'{n:12s} 아직 없음 (export 진행 중 — [8] 확인)')

n_done = sum(os.path.exists(f'{D}/s2/{n}_s2_20band.tif') for n in ALL)
print(f'\ndata/s2 에 있는 신규 지역: {n_done}/7')

Mounted at /content/drive
inje         이동 완료(이전)     480 MB
pyeongchang  이동 완료(이전)     489 MB
gangneung    이동 완료(이전)     465 MB
bonghwa      이동 완료(이전)     490 MB
geumsan      지금 이동          372 MB
geochang     이동 완료(이전)     499 MB
wando        지금 이동          399 MB

data/s2 에 있는 신규 지역: 7/7


In [39]:
# ============================================================
# [10] 라벨 생성 — s2 tif 가 있고 라벨이 아직 없는 지역만
# ============================================================
for n in NEW:
    if not os.path.exists(f'{D}/s2/{n}_s2_20band.tif'):
        print(f'{n:12s} s2 없음 — 건너뜀'); continue
    if os.path.exists(f'{D}/{n}_label.tif'):
        print(f'{n:12s} 라벨 있음 — 건너뜀'); continue
    build_labels(n)

inje         라벨 있음 — 건너뜀
pyeongchang  라벨 있음 — 건너뜀
gangneung    라벨 있음 — 건너뜀
bonghwa      라벨 있음 — 건너뜀
  geumsan 라벨 생성 완료  0:191,706  1:1,099,547  2:1,943,548  3:604,038  255:5,594,865
  geochang 라벨 생성 완료  0:287,157  1:2,370,785  2:1,859,628  3:1,003,375  255:7,168,011
  wando 라벨 생성 완료  0:84,381  1:617,362  2:519,095  3:119,107  255:11,656,402


In [40]:
# ============================================================
# [11] 2024+ 학습 가능 픽셀 수 (02 [12]와 동일 기준)
#   유효 = (label 1~3) & (year >= 2024)  — 연도 조건만 쓰면 배경이 섞인다 (19-1)
# ============================================================
for n in OLD + NEW:
    lp, yp = f'{D}/{n}_label.tif', f'{D}/{n}_year.tif'
    if not os.path.exists(lp): print(f'{n:12s} (라벨 미생성)'); continue
    with rasterio.open(lp) as s: lab = s.read(1)
    with rasterio.open(yp) as s: yr  = s.read(1)
    v = (lab >= 1) & (lab <= 3) & (yr >= 2024)
    cls = [int(((lab == k) & v).sum()) for k in (1, 2, 3)]
    print(f'{n:12s} 2024+ {v.sum():>10,}   침엽 {cls[0]:>9,} 활엽 {cls[1]:>9,} 혼효 {cls[2]:>9,}')

daejeon      2024+  1,029,600   침엽   410,192 활엽   410,816 혼효   208,592
hongcheon    2024+  4,737,009   침엽 1,786,371 활엽 2,321,123 혼효   629,515
suncheon     2024+  1,867,787   침엽   968,819 활엽   709,519 혼효   189,449
inje         2024+  3,770,291   침엽 1,694,054 활엽 1,590,472 혼효   485,765
pyeongchang  2024+  3,488,942   침엽 1,536,535 활엽 1,616,565 혼효   335,842
gangneung    2024+  2,977,229   침엽 1,210,231 활엽 1,444,499 혼효   322,499
bonghwa      2024+  3,877,928   침엽 2,331,223 활엽 1,277,061 혼효   269,644
geumsan      2024+    856,618   침엽   277,966 활엽   440,068 혼효   138,584
geochang     2024+  1,526,772   침엽   767,170 활엽   452,443 혼효   307,159
wando        2024+    314,645   침엽   124,725 활엽   181,911 혼효     8,009


In [41]:
# ============================================================
# [12] 정합 검증 — 라벨 경계가 산 능선/계곡을 따라가면 정상
#   봉화: 2025년 3월 산불 인접(영양·안동) — 영상에 탄 자리가 보이는지 확인
# ============================================================
for n in NEW:
    if os.path.exists(f'{D}/{n}_label.tif'):
        check_align(n)

Output hidden; open in https://colab.research.google.com to view.

In [42]:
# ============================================================
# [13] 지역 기록표 → region_meta.csv
#   논문에 지역 선정 기준과 함께 싣는다 (위도·고도·클래스 비율·영상 장수)
# ============================================================
meta = pd.read_csv(f'{D}/screen_result.csv')
if os.path.exists(LOG):
    log = pd.read_csv(LOG).drop_duplicates('name', keep='last')
    meta = meta.merge(log[['name','summer','leafoff']], on='name', how='left')
meta.to_csv(f'{D}/region_meta.csv', index=False)
print(meta.to_string(index=False))

       name   lat    lon aoi_km   conif   broad  mixed   total  conif_pct  larch_pct  evergreen_pct  elev_mean  ok_600k  summer  leafoff
       inje 38.04 128.25  35x35 1685673 1561568 477794 3725035       45.3       19.0           0.00      622.0     True      83      118
  gangneung 37.71 128.83  35x35 1185042 1421360 317692 2924095       40.5       11.0           0.00      434.0     True     180      249
pyeongchang 37.54 128.50  35x35 1516749 1585289 331632 3433670       44.2       45.8           0.00      760.0     True      41       55
    bonghwa 36.92 128.91  35x35 2304358 1257247 264193 3825800       60.2       16.7           0.00      561.0     True     137      185
    geumsan 36.13 127.48  28x33  278007  439896 138604  856508       32.5       19.5           0.00      284.0     True      49       62
   geochang 35.71 127.89  35x35  766066  450554 306367 1522988       50.3       11.7           0.00      550.0     True     100      125
      wando 34.21 126.79  35x35  124024  

In [43]:
# ============================================================
# [14] 전체 범위 점검 — 라벨은 숲인데 영상에 나무가 없는 곳
#   [12]는 중심 12km 만 본다 (좌표 어긋남 확인용).
#   여기서는 지역 전체에서 '2024+ 숲 라벨 & 여름 NDVI < 0.3' 비율을 센다.
#   → 산불·벌채·개발로 라벨과 영상이 다른 곳. 기존 3곳이 기준선.
#   영상 형식: 대전은 float(0–1), 나머지는 int16(×10000) — 읽을 때 맞춘다
#   그림: 전체 범위를 1/8 로 줄여서 의심 픽셀을 빨갛게 표시 (SHOW_FIG=True 일 때)
# ============================================================
import matplotlib.pyplot as plt

NDVI_TH  = 0.3
SHOW_FIG = False
rows = []
for n in OLD + NEW:
    s2p, lp, yp = f'{D}/s2/{n}_s2_20band.tif', f'{D}/{n}_label.tif', f'{D}/{n}_year.tif'
    if not (os.path.exists(s2p) and os.path.exists(lp)):
        print(f'{n:12s} (s2 또는 라벨 없음 — 건너뜀)'); continue
    with rasterio.open(s2p) as s:
        scale = 10000.0 if s.dtypes[0] == 'int16' else 1.0
        red = s.read(3).astype('float32') / scale      # s_B4 (여름)
        nir = s.read(7).astype('float32') / scale      # s_B8 (여름)
    with rasterio.open(lp) as s: lab = s.read(1)
    with rasterio.open(yp) as s: yr  = s.read(1)
    ok   = np.isfinite(red) & np.isfinite(nir) & ((red + nir) > 0)   # 영상 결측 제외
    ndvi = np.full(red.shape, np.nan, dtype='float32')
    ndvi[ok] = (nir[ok] - red[ok]) / (nir[ok] + red[ok])
    forest = (lab >= 1) & (lab <= 3) & (yr >= 2024) & ok
    bad    = forest & (ndvi < NDVI_TH)
    nodata = int(((lab >= 1) & (lab <= 3) & (yr >= 2024) & ~ok).sum())
    rows.append(dict(name=n, forest_px=int(forest.sum()), no_image_px=nodata,
                     bad_px=int(bad.sum()),
                     bad_pct=round(bad.sum() / max(forest.sum(), 1) * 100, 3),
                     ndvi_median=round(float(np.nanmedian(ndvi[forest])), 3)))

    if SHOW_FIG and n in NEW:
        k = 8
        rgb = np.stack([red[::k, ::k], nir[::k, ::k], red[::k, ::k]], -1)
        rgb = np.clip((rgb - 0.02) / 0.4, 0, 1)
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(rgb)
        yy, xx = np.nonzero(bad[::k, ::k])
        ax.scatter(xx, yy, s=1, c='red')
        ax.set_title(f'{n}  bad {rows[-1]["bad_pct"]}%  (red = forest label, NDVI<{NDVI_TH})')
        ax.axis('off'); plt.show()
    del red, nir, lab, yr, ndvi, forest, bad

chk14 = pd.DataFrame(rows)
print(chk14.to_string(index=False))
chk14.to_csv(f'{D}/label_image_mismatch.csv', index=False)

       name  forest_px  no_image_px  bad_px  bad_pct  ndvi_median
    daejeon    1029600            0    6856    0.666        0.870
  hongcheon    4736929           80    2619    0.055        0.878
   suncheon    1867787            0     784    0.042        0.868
       inje    3726307        43984    3874    0.104        0.881
pyeongchang    3435235        53707     518    0.015        0.880
  gangneung    2925289        51940    6262    0.214        0.878
    bonghwa    3826667        51261     994    0.026        0.874
    geumsan     856618            0    1060    0.124        0.886
   geochang    1522975         3797    1350    0.089        0.871
      wando     311791         2854     273    0.088        0.889
